# Lab 6.3 &mdash; Adequacy, Re-querying and Multi-Hop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Judge your own retrieval &mdash; does it actually contain what was asked for?
- Re-query with a term the first hop taught you
- Follow a chain across three hops without inventing a fourth
- Stop: a hop budget, a repeat detector, and &lsquo;I could not find it&rsquo; as a real outcome

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **This is what makes it agentic.** A pipeline retrieves once. Everything in this lab
> is the loop that a pipeline cannot have, and the stops that keep it from running away.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents. Read 3.2: the rule and the exception that qualifies it are
# adjacent sentences, which is the whole of Lab 6.1's first lesson. Note also what is NOT
# here -- there is nothing about FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """
## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """
## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.1 (nothing to fill in)
import re

SECTION_RE = re.compile(r"^##\s+(.*)$", re.M)
STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def terms(text):
    return {w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1}

def chunk_by_section(source, text):
    out, parts = [], SECTION_RE.split(text)
    for i in range(1, len(parts) - 1, 2):
        heading, body = parts[i].strip(), " ".join(parts[i + 1].split())
        out.append({"source": source, "section": heading, "text": heading + " -- " + body})
    return out

INDEX = [c for source, text in DOCS.items() for c in chunk_by_section(source, text)]

def similarity(query, chunk):
    q = terms(query)
    return len(q & terms(chunk["text"])) / len(q) if q else 0.0

def search(query, index=None, k=4, floor=0.0):
    index = INDEX if index is None else index
    scored = sorted(((similarity(query, c), c) for c in index), key=lambda sc: -sc[0])
    return [{"score": round(s, 3), **c} for s, c in scored[:k] if s >= floor]

print(f"index: {len(INDEX)} chunks")

## Concept

The first retrieval usually returns something. The question is whether it returns *enough*, and
that is answerable without a model: **did what came back contain the thing the question asked
about?**

When it did not, the results still tell you something &mdash; they hand you the corpus's own
vocabulary, which is exactly what the second query needed.

## Section 1 &mdash; Was that enough?

An adequacy test that is honest has to be able to say no. Test it on a retrieval you know is
inadequate before you trust it on one you do not.

In [ ]:
def adequate(question: str, results: list, need_terms=None) -> bool:
    """True if the retrieved text covers what the question is about.

    `need_terms` names the things that must appear; when it is None, fall back to the
    question's own content words.
    """
    if not results:
        return False
    need = set(t.lower() for t in need_terms) if need_terms else terms(question)
    covered = terms(" ".join(r["text"] for r in results))
    return need <= covered


def missing_terms(question: str, results: list, need_terms=None) -> set:
    """What the question asked about that the results never mention."""
    need = set(t.lower() for t in need_terms) if need_terms else terms(question)
    return need - terms(" ".join(r["text"] for r in results))

In [ ]:
# --- Self-check: Section 1
check("an empty retrieval is never adequate",
      lambda: adequate("anything at all", []) is False)
check("a retrieval that covers the asked-for term is adequate",
      lambda: adequate("sanctions", search("sanctions review", k=2), need_terms=["sanctions"])
              is True)
check("one that does not is NOT adequate, even though it returned rows",
      lambda: adequate("hedging", search("FX hedging policy", k=4), need_terms=["hedging"])
              is False,
      "four chunks came back and none of them is about hedging -- a length check would pass this")
check("and it names what was missing",
      lambda: "hedging" in missing_terms("hedging", search("FX hedging policy", k=4),
                                         need_terms=["hedging"]))
check("nothing is missing from an adequate retrieval",
      lambda: missing_terms("sanctions", search("sanctions review", k=2),
                            need_terms=["sanctions"]) == set())
check("the missing term is what the next query should be about",
      lambda: len(missing_terms("r04", search("invalid iban", k=2), need_terms=["r04"])) <= 1)

## Section 2 &mdash; The chain

Three hops, and the point is that hop two's query contains a word you could not have known before
hop one ran. That is what &ldquo;multi-hop&rdquo; means &mdash; not three searches, but three searches where
each one is written from the last one's answer.

In [ ]:
CODE_RE = re.compile(r"\b(R\d{2}|[A-Z]{2,}_[A-Z_]+)\b")

def follow_up(results: list, asked: str):
    """A new query built from a term the results just taught you, or None if they taught nothing."""
    found = []
    for r in results:
        found += CODE_RE.findall(r["text"])
    fresh = [f for f in found if f.lower() not in (asked or "").lower()]
    return fresh[0] if fresh else None


def multi_hop(question: str, max_hops: int = 3) -> dict:
    """Retrieve, read what came back, and ask again with what it taught you."""
    asked, hops, seen = question, [], set()
    for _ in range(max_hops):
        results = search(asked, k=2)
        hops.append({"query": asked, "sections": [r["section"] for r in results]})
        nxt = follow_up(results, asked)
        if nxt is None:
            return {"hops": hops, "outcome": "exhausted"}
        if nxt in seen:
            return {"hops": hops, "outcome": "repeat"}
        seen.add(nxt)
        asked = nxt
    return {"hops": hops, "outcome": "budget"}

In [ ]:
# --- Self-check: Section 2
check("the first hop on a beneficiary question finds section 3.3",
      lambda: multi_hop("what happens with a wrong beneficiary iban")["hops"][0]["sections"][0]
              .startswith("3.3"))
check("hop two asks about a code hop one taught it",
      lambda: multi_hop("what happens with a wrong beneficiary iban")["hops"][1]["query"]
              in ("R04", "INVALID_IBAN"),
      "that term was not in the question and could not have been -- the corpus supplied it")
check("a question whose answer names no codes stops instead of inventing one",
      lambda: multi_hop("who may approve a release up to 250,000")["outcome"] == "exhausted")
check("the run always reports why it stopped",
      lambda: multi_hop("what happens with a wrong beneficiary iban")["outcome"]
              in ("exhausted", "repeat", "budget"))
check("the hop budget is respected",
      lambda: len(multi_hop("what happens with a wrong beneficiary iban", max_hops=2)["hops"]) <= 2)
check("a term already in the query is not chased again",
      lambda: follow_up(search("INVALID_IBAN", k=2), "INVALID_IBAN") != "INVALID_IBAN")
check("nothing new to chase returns None rather than an empty string",
      lambda: follow_up([{"text": "no codes here at all"}], "x") is None)

def _chain():
    out = multi_hop("what happens with a wrong beneficiary iban")
    for i, h in enumerate(out["hops"], 1):
        print(f"  hop {i}: {h['query'][:48]:50} -> {h['sections']}")
    print("  stopped:", out["outcome"])
guard(_chain)

## Section 3 &mdash; Retrieve, judge, retry

Now put Section 1 and Section 2 together: retrieve, ask whether it was enough, and if it was not,
try again with the corpus's own words &mdash; under a budget, and reporting failure honestly.

In [ ]:
def answer_with_retry(question: str, need_terms=None, max_tries: int = 3) -> dict:
    """Retrieve until adequate, or until the budget runs out. Never pretends."""
    tried, query = [], question
    for attempt in range(max_tries):
        results = search(query, k=3)
        tried.append(query)
        if adequate(question, results, need_terms):
            return {"outcome": "answered", "query": query, "results": results,
                    "attempts": attempt + 1}
        nxt = follow_up(results, query)
        if nxt is None or nxt in tried:
            break
        query = nxt
    return {"outcome": "not found", "query": query, "results": [],
            "attempts": len(tried),
            "missing": sorted(missing_terms(question, search(question, k=3), need_terms))}

In [ ]:
# --- Self-check: Section 3
check("an answerable question is answered",
      lambda: answer_with_retry("what does a sanctions review need",
                                need_terms=["sanctions", "compliance"])["outcome"] == "answered")
check("and it did not need many attempts",
      lambda: answer_with_retry("what does a sanctions review need",
                                need_terms=["sanctions", "compliance"])["attempts"] <= 2)
check("an unanswerable question ends as 'not found', not as a wrong answer",
      lambda: answer_with_retry("what is the JPY hedging policy",
                                need_terms=["hedging"])["outcome"] == "not found")
check("and it says what was missing",
      lambda: "hedging" in answer_with_retry("what is the JPY hedging policy",
                                             need_terms=["hedging"])["missing"],
      "'the corpus has nothing on hedging' is a useful answer; 'I don't know' is not")
check("it returns no results when it did not find them",
      lambda: answer_with_retry("what is the JPY hedging policy",
                                need_terms=["hedging"])["results"] == [],
      "handing back the four irrelevant chunks anyway is how a refusal becomes a hallucination")
check("the attempt count never exceeds the budget",
      lambda: answer_with_retry("what is the JPY hedging policy", need_terms=["hedging"],
                                max_tries=2)["attempts"] <= 2)

def _both():
    for q, need in (("what does a sanctions review need", ["sanctions", "compliance"]),
                    ("what is the JPY hedging policy", ["hedging"])):
        out = answer_with_retry(q, need_terms=need)
        extra = f"  missing: {out.get('missing')}" if out["outcome"] == "not found" else ""
        print(f"  {out['outcome']:10} in {out['attempts']} attempt(s)  {q[:38]}{extra}")
guard(_both)

## Run it for real

Let the model judge adequacy instead of the term check, on the same two questions. The one to
watch is the second: a model asked &ldquo;is this enough?&rdquo; about four irrelevant chunks has every
incentive to say yes.

In [ ]:
if llm_ready():
    def _judge():
        for q in ("what does a sanctions review need", "what is the JPY hedging policy for us"):
            results = search(q, k=3)
            context = "\n".join(f"- [{r['section']}] {r['text'][:150]}" for r in results)
            verdict = ask(f"Question: {q}\n\nRetrieved:\n{context}\n\n"
                          "Can this question be answered from the retrieved text alone? "
                          "Reply YES or NO, then one short sentence.",
                          system="Begin your reply with YES or NO.")
            print(f"  {q}")
            print(f"      model: {verdict.strip()[:150]}")
            print(f"      term check: {'adequate' if adequate(q, results) else 'not adequate'}")
            print()
    guard(_judge)

### Read it

If the model says YES to the FX question, you have watched the failure this lab exists to prevent:
the retrieval was inadequate, the judge was the same kind of thing that will write the answer, and
nothing stopped it.

The term check is crude and cannot be talked round. In production you want both &mdash; the cheap
mechanical check as a floor, and the model for the judgements the check is too blunt to make.

In [ ]:
score()

## Your turn

1. `adequate` requires *every* term. Make it a fraction &mdash; three-quarters covered is enough &mdash;
   and find the question where that change gives you a confident wrong answer.
2. `follow_up` chases reason codes because that is what this corpus is made of. What is the
   equivalent handle in your corpus &mdash; a ticket id, a product code, a section number? Write the
   regex and see how far a chain gets.
3. Give `multi_hop` a wall-clock deadline as well as a hop budget, then make one search slow.
   Which stop fires first, and which one would you actually have wanted?